## This code will implement the Seq2seq model

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
import pickle
from datasets import load_from_disk

c:\Users\HLM\Desktop\Diverse projects\Stanford Course\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pipeline creation

In [126]:
dataset = load_from_disk("Dataset_Seq2seq")

with open("Mapping.pkl", "rb") as f:
    mapping = pickle.load(f)

In [127]:
dataset.set_format(type="torch", columns=["context", "text"])
split = dataset.train_test_split(test_size=0.2,train_size=0.8, seed=42)

train = split["train"]
test = split["test"]

In [128]:
class SeqDataset(Dataset) :
    def __init__(self, data, mapping) :
        self.data = data
        self.length = len(data)
        self.mapping = mapping

    def __len__(self) :
        return self.length

    def __getitem__(self, i) :
        context = [self.mapping[char] for char in self.data["context"][i]]
        text = [self.mapping[char] for char in self.data["text"][i]]
        return (context,text)

In [129]:
train_data = SeqDataset(train,mapping)
test_data = SeqDataset(test,mapping)

In [133]:
def collate_fn(batch) :
    contexts = [torch.tensor(item[0]) for item in batch]
    texts = [torch.tensor(item[1]) for item in batch]

    #Padding and return length for decodeur so context
    length_context = [len(item) for item in contexts]
    length_text = [len(item) for item in texts]
    contexts_padded = pad_sequence(contexts, batch_first=True, padding_value=0)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
        
    return (contexts_padded, texts_padded), (torch.tensor(length_context), torch.tensor(length_text))

In [134]:
train_ld = DataLoader(train_data, batch_size=2, shuffle=True, drop_last=True, collate_fn=collate_fn)
test_ld = DataLoader(test_data, batch_size=2, shuffle=True, drop_last=True, collate_fn=collate_fn)

List of operations :

- First the entire batch pass through the Encodeur, we extract the h,c from this

- Then we iterate over all the length of the sequence, with the previous input and we use at first the h,c from the Encodeur

- Then we calculate the loss

## Create Seq2seq

In [136]:
class Encodeur(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super(Encodeur, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(input_size = embedding_dim, 
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            batch_first=True)

    def forward(self, x, length_batch):
        #Encodeur
        emb_x = self.embed(x)
        #Pack before LSTM to avoid unecessary compute
        packed_x = pack_padded_sequence(emb_x, length_batch, batch_first=True, enforce_sorted=False)                      
        packed_out, (h,c) = self.lstm(packed_x)             
        #Unpack after
        _, _ = pad_packed_sequence(packed_out, batch_first=True) #First right now only h,c
        return (h, c)

In [137]:
class Decodeur(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super(Decodeur, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(input_size = embedding_dim, 
                            hidden_size = hidden_size, 
                            num_layers = num_layers, 
                            batch_first=True)
        
        self.final = nn.Linear(hidden_size,vocab_size)

    def forward(self, x, h, c, length_batch):
        emb_x = self.embed(x)
        #Pack before LSTM to avoid unecessary compute
        packed_x = pack_padded_sequence(emb_x, length_batch, batch_first=True, enforce_sorted=False)                      
        x_lstm, (h,c) = self.lstm(packed_x, (h,c))             
        #Unpack 
        unpacked_out,_ = pad_packed_sequence(x_lstm, batch_first=True) 
        logits = self.final(unpacked_out)
  
        return logits, (h, c)

In [ ]:
vocab_size = len(mapping)
embedding_size = 100
hidden_size = 150

enco = Encodeur(vocab_size, embedding_size, hidden_size)
deco = Decodeur(vocab_size, embedding_size, hidden_size)

loss_fn = nn.CrossEntropyLoss(ignore_index=0)

## Creation boucle entraînement Decodeur

In [ ]:
for (context, text), (length_cont, length_text) in train_ld :

    h_enco,c_enco = enco(context,length_cont)

    output = []
    h_dec = h_enco
    c_dec = c_enco

    for j in range(len(text[0])) :
        inp = text[:,j].unsqueeze(1)
        logits, (h_dec, c_dec) = deco(inp,h_dec,c_dec,length_text)
        output.append(logits)

    outputs = torch.cat(output, dim=1)
    loss_fn(outputs.view(-1,vocab_size),batch_text.view(-1))

tensor([[112, 169],
        [113, 169]])
